# 🔦 GASMAS Simulation - Lung Tissues + Thorax

**Technique:** GASMAS (Gas in Scattering Media Absorption Spectroscopy)  
**Illumination Mode:** External Illumination  
**Gas:** O₂ at 763.84 nm (A-band), scan ±0.025 nm  
**Material:** Lung Tissues + Thorax - 8×8×6 cm, 85% porosity  
**Conditions:** T=296K, P=1 atm, 21% O₂  
**SDS scan:** 5, 10, 15, 20, 30,35 mm source-detector separations

---
### Pipeline
```
HITRAN/HAPI → O2 absorption spectrum α(λ)
MCX         → fluence + photon pathlength in air pockets → GASMAS
```
### ⚠️ Enable T4 GPU: Runtime → Change runtime type → T4 GPU

## Cell 1 — Install Dependencies

In [ ]:
!pip install pmcx -q

In [ ]:
pip install hitran-api

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.stats import linregress
from scipy.interpolate import interp1d
import time, os
import warnings

# Suppress specific warnings early
warnings.filterwarnings("ignore", category=DeprecationWarning, module='matplotlib')
warnings.filterwarnings("ignore", category=UserWarning, module='matplotlib')
warnings.filterwarnings("ignore", category=FutureWarning, module='pyparsing')
warnings.filterwarnings("ignore", category=DeprecationWarning, module='pyparsing')

import pmcx
print('pmcx version:', pmcx.__version__)

try:
    gpu_info = pmcx.gpuinfo()
    print('GPU:', gpu_info)
except:
    print('No GPU — running on CPU')

# HAPI — HITRAN Application Programming Interface
import hapi
print('HAPI imported ✓')


## Cell 2 — Download O₂ Line Data from HITRAN via HAPI

Downloads the O₂ A-band lines near 763.84 nm from the HITRAN database.  

In [ ]:
# ── Spectral range ────────────────────────────────────────────
LAMBDA_CENTER_NM = 763.84251         # O2 A-band peak (nm)
SCAN_HALF_NM     = 0.025           # ±0.025 nm scan range

lambda_min_nm = LAMBDA_CENTER_NM - SCAN_HALF_NM   # 763.815 nm
lambda_max_nm = LAMBDA_CENTER_NM + SCAN_HALF_NM   # 763.865 nm

# Convert to wavenumber (cm⁻¹): ν = 1e7 / λ_nm
nu_min = 1e7 / lambda_max_nm   # smaller λ → larger ν
nu_max = 1e7 / lambda_min_nm
NU_STEP = 0.0001   # cm⁻¹
print(f'Wavelength range : {lambda_min_nm:.4f} – {lambda_max_nm:.4f} nm')
print(f'Wavenumber range : {nu_min:.4f} – {nu_max:.4f} cm⁻¹')

# ── HAPI setup ────────────────────────────────────────────────
HAPI_DIR = './hapi_data'
os.makedirs(HAPI_DIR, exist_ok=True)
hapi.db_begin(HAPI_DIR)

# Molecule ID 7 = O2, Isotopologue 1 = 16O2
# Adds a small buffer (±2 cm⁻¹) to capture wing contributions
print('\nDownloading O2 line data from HITRAN...')
hapi.fetch('O2_Aband', 7, 1,
           nu_min - 2.0,
           nu_max + 2.0)
print('Download complete ✓')

print(hapi.tableList())

## Cell 3 — Compute O₂ Absorption Spectrum (Voigt profile via HAPI)

Conditions: T=296 K, P=1 atm, O₂ concentration = 21%

In [ ]:
# ── Gas conditions ────────────────────────────────────────────
T_GAS     = 296.0    # K
P_ATM     = 1.0      # atm
P_HAPI    = P_ATM    # HAPI uses atm
X_O2      = 0.21     # O2 mole fraction in air

# Number density of O2 at T, P  (molecules/cm³)
# n = P * X_O2 * N_A / (R * T)  in SI, then convert to cm⁻³
N_A  = 6.02214e23    # mol⁻¹
R    = 82.057        # cm³·atm/(mol·K)
n_O2 = P_ATM * X_O2 * N_A / (R * T_GAS)   # molecules/cm³
print(f'O2 number density: {n_O2:.4e} molecules/cm³')

# ── High-resolution wavenumber grid ──────────────────────────
N_POINTS  = 5000
nu_grid   = np.linspace(nu_min, nu_max, N_POINTS)   # cm⁻¹
dnu       = nu_grid[1] - nu_grid[0]
print(f'Spectral grid    : {N_POINTS} points, step = {dnu*1000:.5f} m-cm⁻¹')

# ── Calculate absorption cross-section σ(ν) via HAPI ─────────
# absorptionCoefficient_Voigt returns: (nu_out, coeff)  in cm⁻¹
# coeff = α(ν) = σ(ν) × n  [cm⁻¹]
nu_out, alpha_cm = hapi.absorptionCoefficient_Voigt(
    SourceTables="O2_Aband",
    Diluent={'air': 1.0},          # air-broadening from HITRAN
    Environment={'p': P_ATM, 'T': T_GAS},
    HITRAN_units=False,             # output in cm⁻¹ directly
    WavenumberRange=[nu_min, nu_max],
    WavenumberStep=NU_STEP,         # 0.0001 cm⁻¹
    WavenumberWing=50,              # 50 cm⁻¹ wing cutoff
)

# α(ν) is in cm⁻¹ — convert to mm⁻¹ for MCX unit consistency
alpha_nu_mm = np.array(alpha_cm) * 0.1   # cm⁻¹ → mm⁻¹
alpha_nu_mm=alpha_nu_mm*X_O2

# Convert wavenumber axis back to wavelength (nm)
nu_arr    = np.array(nu_out)
lambda_nm = 1e7 / nu_arr   # nm  (note: reverses order)

# Keep in ascending wavelength order
sort_idx      = np.argsort(lambda_nm)
lambda_nm     = lambda_nm[sort_idx]
alpha_nu_mm   = alpha_nu_mm[sort_idx]

# Interpolator for use later
alpha_interp = interp1d(lambda_nm, alpha_nu_mm,
                        kind='linear', fill_value=0, bounds_error=False)

print(f'\nPeak absorption  : {alpha_nu_mm.max():.5f} mm⁻¹')
print(f'Peak wavelength  : {lambda_nm[np.argmax(alpha_nu_mm)]:.5f} nm')

# ── Plot spectrum ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lambda_nm, alpha_nu_mm * 10, 'b-', lw=1.2)   # ×10 → cm⁻¹ display
ax.axvline(LAMBDA_CENTER_NM, color='r', ls='--', lw=1.2,
           label=f'Centre {LAMBDA_CENTER_NM} nm')
ax.set_xlabel('Wavelength (nm)', fontsize=12)
ax.set_ylabel('Absorption coeff α (cm⁻¹)', fontsize=12)
ax.set_title(f'O₂ A-band Absorption Spectrum — HITRAN Voigt Profile\n'
             f'T={T_GAS}K, P={P_ATM}atm, X_O2={X_O2}', fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(lambda_min_nm, lambda_max_nm)
plt.ticklabel_format(useOffset=False, style='plain', axis='x')
plt.tight_layout()
#plt.savefig('o2_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()
print('O2 spectrum computed ✓')

## Cell 4 — Simulation Parameters & Geometry

In [ ]:
# ── Cell 4 — Parameters (TWO-LAYER: skin + styrofoam) ─────────

VOX_SIZE = 0.1       # mm
PHYS_X   = 80.0      # mm  (8 cm)
PHYS_Y   = 80.0      # mm  (8 cm)
PHYS_Z   = 75.0      # mm  (6 cm total = 2cm skin + 4cm lung)            ############this

SKIN_THICKNESS_MM  = 35.0   # mm  (2 cm)
FOAM_THICKNESS_MM  = 40.0   # mm  (4 cm)

NX = int(PHYS_X / VOX_SIZE)   # 800
NY = int(PHYS_Y / VOX_SIZE)   # 800
NZ = int(PHYS_Z / VOX_SIZE)   # 700  ← increased from 400

# Voxel layer boundaries
SKIN_Z_START = 0
SKIN_Z_END   = int(SKIN_THICKNESS_MM / VOX_SIZE)   # voxels 0–199
FOAM_Z_START = SKIN_Z_END                           # voxels 200–599
FOAM_Z_END   = NZ

POROSITY = 0.85
SEED     = 42
np.random.seed(SEED)

print(f'Grid      : {NX}×{NY}×{NZ}')
print(f'Physical  : {PHYS_X}×{PHYS_Y}×{PHYS_Z} mm')
print(f'Skin layer: z = 0 – {SKIN_Z_END} voxels  '
      f'({SKIN_Z_START*VOX_SIZE}–{SKIN_Z_END*VOX_SIZE} mm)')
print(f'Foam layer: z = {FOAM_Z_START} – {FOAM_Z_END} voxels  '
      f'({FOAM_Z_START*VOX_SIZE}–{FOAM_Z_END*VOX_SIZE} mm)')
print(f'Label RAM : ~{NX*NY*NZ/1e9:.2f} GB')
print(f'Fluence   : ~{NX*NY*NZ*4/1e9:.2f} GB')

# ── Optical properties at 763 nm ──────────────────────────────
# Format: [mua (mm⁻¹), mus_total (mm⁻¹), g, n]
#
# Label 0 = background
# Label 1 = polystyrene solid
# Label 2 = air pockets
# Label 3 = skin tissue  ← NEW
ALPHA_CENTRE= float(alpha_interp(LAMBDA_CENTER_NM))

PROPS = np.array([
    [0,           0,      1.00,  1.00],   # 0 = background
    [0.010,      20.0,   0.90,  1.59],   # 1 = lung tissues
    [ALPHA_CENTRE, 0.10,  0.90,  1.00],  # 2 = air pockets (O2 absorption)
    [0.01,           10.0,   0.90,  1.40],  # 3 skin tissue ← NEW
], dtype=float)

print()
print('Optical properties at 763 nm:')
names = ['Background', 'Polystyrene', 'Air pockets', 'Skin tissue']
for i, (name, p) in enumerate(zip(names, PROPS)):
    print(f'  Label {i} ({name:12s}): '
          f'mua={p[0]:.5f}  mus={p[1]:.3f}  g={p[2]}  n={p[3]}')

# ── GASMAS / source config ────────────────────────────────────
SDS_LIST_MM   = [5, 10, 15, 20,25, 30,35]
BEAM_SIGMA_MM = 1.0
NPHOTONS      = 10_000_000

print(f'\nSDS values : {SDS_LIST_MM} mm')
print(f'Beam sigma : {BEAM_SIGMA_MM} mm')
print(f'Photons    : {NPHOTONS:,}')

## Cell 5 — Build Porous Geometry
> Takes ~3–5 minutes for 800×800×400 grid

In [ ]:
# ── Cell 5 — Build Two-Layer Porous Volume ────────────────────
#
# Label 3 fills skin layer  (z = 0 to SKIN_Z_END)
# Label 1 fills foam layer  (z = SKIN_Z_END to NZ)
# Label 2 = air pockets placed ONLY inside foam layer

def build_two_layer_volume(nx, ny, nz,
                            skin_z_end,
                            porosity,
                            min_r=1, max_r=2,
                            seed=42):
    np.random.seed(seed)

    # Start: skin everywhere
    vol = np.zeros((nx, ny, nz), dtype=np.uint8)

    # z < skin_z_end → label 3 (skin)
    vol[:, :, :skin_z_end] = 3

    # z >= skin_z_end → label 1 (solid polystyrene)
    vol[:, :, skin_z_end:] = 1

    # Place air pockets ONLY in foam region
    foam_voxels = nx * ny * (nz - skin_z_end)
    target_air  = int(porosity * foam_voxels)
    current_air = 0
    attempts    = 0
    t0          = time.time()

    print(f'Building two-layer volume: {nx}×{ny}×{nz}')
    print(f'  Skin  : z = 0 – {skin_z_end}  ({skin_z_end*VOX_SIZE:.0f} mm)')
    print(f'  Foam  : z = {skin_z_end} – {nz}  ({(nz-skin_z_end)*VOX_SIZE:.0f} mm)')
    print(f'  Target air voxels in foam: {target_air:,}  ({porosity*100:.0f}%)')

    while current_air < target_air:
        r  = np.random.randint(min_r, max_r + 1)

        cz_min = skin_z_end + r
        cz_max = nz - r
        if cz_min >= cz_max:
            continue

        cx = np.random.randint(r, nx - r)
        cy = np.random.randint(r, ny - r)
        cz = np.random.randint(cz_min, cz_max)

        zz, yy, xx = np.ogrid[-r:r+1, -r:r+1, -r:r+1]
        sphere = (xx**2 + yy**2 + zz**2) <= r**2

        region = vol[cx-r:cx+r+1, cy-r:cy+r+1, cz-r:cz+r+1]
        current_air += int(np.sum((region == 1) & sphere))
        region[sphere] = 2
        attempts += 1

        if attempts % 50000 == 0:
            print(f'  {current_air/target_air*100:.1f}%  '
                  f'({attempts:,} spheres placed)')

    elapsed = time.time() - t0
    n_skin  = int(np.sum(vol == 3))
    n_solid = int(np.sum(vol == 1))
    n_air   = int(np.sum(vol == 2))
    actual_p = n_air / foam_voxels * 100

    print(f'\nDone in {elapsed:.1f}s')
    print(f'  Skin voxels  : {n_skin:,}')
    print(f'  Solid voxels : {n_solid:,}')
    print(f'  Air voxels   : {n_air:,}  (porosity = {actual_p:.2f}%)')
    return vol

vol = build_two_layer_volume(
    NX, NY, NZ,
    skin_z_end=SKIN_Z_END,
    porosity=POROSITY,
    min_r=1, max_r=2,
    seed=SEED
)

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

# 4-color map: background, solid, air, skin
cmap_geo = mcolors.ListedColormap([
    '#f5f5f5',   # 0 background — white
    '#c8a96e',   # 1 solid PS   — tan
    '#7ec8e3',   # 2 air pocket — light blue
    '#e8a0a0',   # 3 skin       — pink/salmon
])

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Full cross-sections
slices_full = [
    (vol[:, :, SKIN_Z_END + (NZ-SKIN_Z_END)//2].T,
     f'XY — z={( SKIN_Z_END+(NZ-SKIN_Z_END)//2)*VOX_SIZE:.0f}mm (foam mid)',
     'x (mm)', 'y (mm)',
     [0, PHYS_X, 0, PHYS_Y]),

    (vol[:, NY//2, :].T,
     f'XZ — y={NY//2*VOX_SIZE:.0f}mm (centre)',
     'x (mm)', 'z (mm)',
     [0, PHYS_X, 0, PHYS_Z]),

    (vol[NX//2, :, :].T,
     f'YZ — x={NX//2*VOX_SIZE:.0f}mm (centre)',
     'y (mm)', 'z (mm)',
     [0, PHYS_Y, 0, PHYS_Z]),
]

for ax, (sl, title, xl, yl, ext) in zip(axes[0], slices_full):
    im = ax.imshow(sl, origin='lower', cmap=cmap_geo,
                   vmin=0, vmax=3, extent=ext,
                   interpolation='nearest', aspect='auto')
    ax.set_title(title, fontsize=16) # Increased title fontsize
    ax.set_xlabel(xl, fontsize=14); ax.set_ylabel(yl, fontsize=14) # Increased label fontsize
    ax.tick_params(axis='both', which='major', labelsize=12) # Increased tick labelsize

    # Draw skin/foam boundary line on XZ and YZ
    if 'XZ' in title or 'YZ' in title:
        ax.axhline(SKIN_THICKNESS_MM, color='yellow',
                   lw=1.5, ls='--', label='skin/foam boundary')
        ax.legend(fontsize=12, loc='upper right') # Increased legend fontsize

#Zoomed XZ — shows both layers clearly
ZOOM_X_MM  = 20
ZOOM_X_VOX = int(ZOOM_X_MM / VOX_SIZE)
cx = NX // 2

# Left zoom: near-surface (skin + top of foam)
crop1 = vol[cx-ZOOM_X_VOX//2:cx+ZOOM_X_VOX//2, NY//2, :int(30/VOX_SIZE)].T
axes[1,0].imshow(crop1, origin='lower', cmap=cmap_geo,
                 vmin=0, vmax=3,
                 extent=[0, ZOOM_X_MM, 0, 30],
                 interpolation='nearest', aspect='auto')
axes[1,0].axhline(SKIN_THICKNESS_MM, color='yellow', lw=1.5, ls='--')
axes[1,0].set_title('XZ zoom — skin/foam interface (0–30mm depth)', fontsize=12) # Increased title fontsize
axes[1,0].set_xlabel('x (mm)', fontsize=14); axes[1,0].set_ylabel('z (mm)', fontsize=14) # Increased label fontsize
axes[1,0].tick_params(axis='both', which='major', labelsize=12) # Increased tick labelsize

# Centre zoom: mid foam
z_mid_foam = SKIN_Z_END + (NZ - SKIN_Z_END) // 2
crop2 = vol[cx-ZOOM_X_VOX//2:cx+ZOOM_X_VOX//2,
            NY//2,
            z_mid_foam-ZOOM_X_VOX//2:z_mid_foam+ZOOM_X_VOX//2].T
axes[1,1].imshow(crop2, origin='lower', cmap=cmap_geo,
                 vmin=0, vmax=3,
                 extent=[0, ZOOM_X_MM, 0, ZOOM_X_MM],
                 interpolation='nearest', aspect='auto')
axes[1,1].set_title(f'XZ zoom', fontsize=16) # Increased title fontsize
axes[1,1].set_xlabel('x (mm)', fontsize=14); axes[1,1].set_ylabel('z (mm)', fontsize=16) # Increased label fontsize
axes[1,1].tick_params(axis='both', which='major', labelsize=14) # Increased tick labelsize

# Right: layer thickness bar chart
ax = axes[1,2]
ax.barh(['Skin layer', 'Styrofoam'],
        [SKIN_THICKNESS_MM, FOAM_THICKNESS_MM],
        color=['#e8a0a0', '#c8a96e'], edgecolor='gray', height=0.5)
ax.set_xlabel('Thickness (mm)', fontsize=14) # Increased label fontsize
ax.set_title('Layer Thicknesses', fontsize=16) # Increased title fontsize
for i, v in enumerate([SKIN_THICKNESS_MM, FOAM_THICKNESS_MM]):
    ax.text(v + 0.5, i, f'{v:.0f} mm', va='center', fontsize=12) # Increased text fontsize
ax.set_xlim(0, max(SKIN_THICKNESS_MM, FOAM_THICKNESS_MM) * 1.3)
ax.grid(True, alpha=0.3, axis='x')
ax.tick_params(axis='both', which='major', labelsize=12) # Increased tick labelsize

# Legends
patches = [
    mpatches.Patch(color='#f5f5f5', label='Background'),
    mpatches.Patch(color='#c8a96e', label='Solid polystyrene'),
    mpatches.Patch(color='#7ec8e3', label='Air pocket'),
    mpatches.Patch(color='#e8a0a0', label='Skin tissue'),
]
fig.legend(handles=patches, loc='upper center', ncol=4,
           fontsize=14, bbox_to_anchor=(0.5, -0.02)) # Increased legend fontsize

plt.suptitle(
    f'Two-Layer Geometry: Skin ({SKIN_THICKNESS_MM:.0f}mm) + '
    f'Styrofoam ({FOAM_THICKNESS_MM:.0f}mm)  |  '
    f'{np.sum(vol==2)/np.sum(vol>=1)*100:.1f}% porosity in foam',
    fontsize=18, y=1.01
)
plt.tight_layout()
plt.savefig('geometry_two_layer.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Skin  voxels : {np.sum(vol==3):,}')
print(f'Solid voxels : {np.sum(vol==1):,}')
print(f'Air   voxels : {np.sum(vol==2):,}')
print(f'Foam porosity: {np.sum(vol==2)/(np.sum(vol==1)+np.sum(vol==2))*100:.2f}%')

## Cell 6 — MCX Simulation (Reflectance Mode)

**Key MCX settings for GASMAS:**
- Source: Gaussian beam at top surface centre
- Multiple ring detectors at different SDS on **same surface (z=1)**
- `outputtype = 'fluence'` gives normalised fluence per voxel
- We also run with `savedetflag = 'dp'` to get **per-photon pathlength** in each medium

> T4 runtime: ~5–10 min for 10M photons

In [ ]:
NPHOTONS = 10_000_000   # 10M photons for good SDS statistics



SRC_X = NX / 2.0
SRC_Y = NY / 2.0
SRC_Z = 1.0


DET_RADIUS_MM  = 2.5 #2.5 mm ditector radius
DET_RADIUS_VOX = DET_RADIUS_MM / VOX_SIZE


# z=2 = one voxel below surface, catches backscattered photons
DET_Z = 2.0

detpos = np.array([
    [SRC_X + sds / VOX_SIZE, SRC_Y, DET_Z, DET_RADIUS_VOX]
    for sds in SDS_LIST_MM
], dtype=float)

print('Detector positions:')
print(f'  Radius : {DET_RADIUS_MM} mm  ({DET_RADIUS_VOX:.0f} voxels)')
print(f'  Z      : {DET_Z} (voxel units)')
for sds, dp in zip(SDS_LIST_MM, detpos):
    in_vol = 'OK' if dp[0] < NX else '❌ OUTSIDE VOLUME'
    print(f'  SDS={sds:2d}mm  x={dp[0]:.0f}  y={dp[1]:.0f}  z={dp[2]:.0f}  '
          f'r={dp[3]:.0f}vox  {in_vol}')

cfg = {
    'vol'         : vol,
    'unitinmm'    : VOX_SIZE,
    'srctype'     : 'gaussian',
    'srcpos'      : [SRC_X, SRC_Y, SRC_Z],
    'srcdir'      : [0, 0, 1],
    'srcparam1'   : [BEAM_SIGMA_MM / VOX_SIZE, 0, 0, 0],
    'detpos'      : detpos,
    'nphoton'     : NPHOTONS,
    'tstart'      : 0,
    'tend'        : 5e-9,
    'tstep'       : 5e-9,
    'prop'        : PROPS,
    'outputtype'  : 'fluence',
    'savedetflag' : 'dp',      #  d=detectorid, p=pathlength
    'seed'        : SEED,
    'isreflect'   : 1,
    'bc'          : 'aaaaaa',
    'autopilot'   : 1,
    'maxdetphoton': NPHOTONS,
}

print(f'\nRunning MCX  ({NPHOTONS:,} photons)...')
t0  = time.time()
res = pmcx.run(cfg)
elapsed = time.time() - t0

fluence = res['flux'][:, :, :, 0].astype(np.float32)
detp    = np.array(res['detp'])

print(f'Done in {elapsed:.1f}s  ({elapsed/60:.1f} min)')
print(f'Fluence shape : {fluence.shape}')
print(f'detp shape    : {detp.shape}')
print(f'  rows = {detp.shape[0]} fields per photon')
print(f'  cols = {detp.shape[1]:,} detected photons')


### File Location

In [ ]:
#np.savez_compressed('gasmas_simulation_results_35mm.npz', fluence=fluence, detp=detp)
#print('Simulation results saved to gasmas_simulation_results.npz')
#from google.colab import files

#files.download('gasmas_simulation_results_35mm.npz')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import numpy as np

cmap_geo = mcolors.ListedColormap([
    '#f5f5f5',   # 0 background — white
    '#c8a96e',   # 1 solid PS   — tan
    '#7ec8e3',   # 2 air pocket — light blue
    '#e8a0a0',   # 3 skin       — pink/salmon
])

phys_x_mm = NX * VOX_SIZE
phys_y_mm = NY * VOX_SIZE
phys_z_mm = NZ * VOX_SIZE

# Top View (XY Plane)
fig, axes = plt.subplots(1, 2, figsize=(18, 8)) # Increased figure width for better legend placement

# Get the top-most slice of the volume (at z=0 voxel index, which is 0 to VOX_SIZE mm depth)
# For visualization, we'll assume this represents the surface.
top_slice_idx = 0 # Corresponds to z = 0 to 0.1mm
top_slice = vol[:, :, top_slice_idx].T # Transpose for correct orientation with imshow

ax_top = axes[0]
im_top = ax_top.imshow(top_slice, origin='lower', cmap=cmap_geo,
                       vmin=0, vmax=3,
                       extent=[0, phys_x_mm, 0, phys_y_mm],
                       interpolation='nearest')
ax_top.set_title('Top View (XY Plane)', fontsize=18) # Increased title fontsize
ax_top.set_xlabel('X (mm)', fontsize=18)
ax_top.set_ylabel('Y (mm)', fontsize=18)
ax_top.grid(True, alpha=0.3)
ax_top.tick_params(axis='both', which='major', labelsize=16) # Increased tick labelsize

# Plot Source on Top View
source_x_mm = SRC_X * VOX_SIZE
source_y_mm = SRC_Y * VOX_SIZE
ax_top.plot(source_x_mm, source_y_mm, 'ro', markersize=10, label='Source', markeredgecolor='black')
ax_top.add_artist(plt.Circle((source_x_mm, source_y_mm), BEAM_SIGMA_MM, color='r', alpha=0.3, fill=True, linestyle='--', linewidth=1))

# Plot Detectors on Top View
for i, sds_val in enumerate(SDS_LIST_MM):
    det_x_voxel = detpos[i, 0]
    det_y_voxel = detpos[i, 1]
    det_radius_voxel = detpos[i, 3]

    det_x_mm = det_x_voxel * VOX_SIZE
    det_y_mm = det_y_voxel * VOX_SIZE
    det_radius_mm = det_radius_voxel * VOX_SIZE

    ax_top.plot(det_x_mm, det_y_mm, 'bo', markersize=8, label=f'Detector {sds_val}mm' if i == 0 else '', markeredgecolor='black')
    #ax_top.add_artist(plt.Circle((det_x_mm, det_y_mm), det_radius_mm, color='b', alpha=0.2, fill=True, linestyle='--', linewidth=1))

# --- Side View (XZ Plane) ---
# Slice at the center of Y-axis
center_y_voxel = NY // 2
side_slice = vol[:, center_y_voxel, :].T

ax_side = axes[1]
im_side = ax_side.imshow(side_slice, origin='upper', cmap=cmap_geo, # Changed origin to 'upper'
                         vmin=0, vmax=3,
                         extent=[0, phys_x_mm, phys_z_mm, 0], # Inverted Z-axis in extent
                         interpolation='nearest', aspect='auto') # Use aspect='auto' for non-square voxels
ax_side.set_title(f'Side View (XZ Plane at Y={center_y_voxel * VOX_SIZE:.1f}mm)', fontsize=18) # Increased title fontsize
ax_side.set_xlabel('X (mm)', fontsize=18)
ax_side.set_ylabel('Z (mm)', fontsize=18)
ax_side.grid(True, alpha=0.3)
ax_side.tick_params(axis='both', which='major', labelsize=16) # Increased tick labelsize

# Plot Source on Side View
# The source is a point in the XZ plane for this view. Its Z is SRC_Z * VOX_SIZE
source_x_mm = SRC_X * VOX_SIZE
source_z_mm = SRC_Z * VOX_SIZE # This is the actual Z depth in the MCX config
ax_side.plot(source_x_mm, source_z_mm, 'ro', markersize=10, label='Source', markeredgecolor='black')
ax_side.add_artist(plt.Circle((source_x_mm, source_z_mm), BEAM_SIGMA_MM, color='r', alpha=0.3, fill=True, linestyle='--', linewidth=1))


# Plot Detectors on Side View
for i, sds_val in enumerate(SDS_LIST_MM):
    det_x_voxel = detpos[i, 0]
    det_z_voxel = detpos[i, 2] # This is DET_Z
    det_radius_voxel = detpos[i, 3]

    det_x_mm = det_x_voxel * VOX_SIZE
    det_z_mm = det_z_voxel * VOX_SIZE
    det_radius_mm = det_radius_voxel * VOX_SIZE

    ax_side.plot(det_x_mm, det_z_mm, 'bo', markersize=8, label=f'Detector {sds_val}mm' if i == 0 else '', markeredgecolor='black')
    # Draw a circle for the detector radius, but only in the X-Z plane it's a circle.
    # For a 2D slice, a 3D sphere projects as a circle.
    #ax_side.add_artist(plt.Circle((det_x_mm, det_z_mm), det_radius_mm, color='b', alpha=0.2, fill=True, linestyle='--', linewidth=1))


# Add a line to indicate the skin/foam boundary
skin_thickness_mm = SKIN_THICKNESS_MM
ax_side.axhline(skin_thickness_mm, color='yellow', linestyle='--', linewidth=2, label='Skin/Foam Boundary')


# Create legend for materials (from 31huqyLaUg-n)
patches = [
    mpatches.Patch(color='#f5f5f5', label='Background'),
    mpatches.Patch(color='#c8a96e', label='Lung tissues'),
    mpatches.Patch(color='#7ec8e3', label='Air pocket'),
    mpatches.Patch(color='#e8a0a0', label='Skin tissue'),
]
fig.legend(handles=patches + [mpatches.Patch(color='r', label='Source'),
                              mpatches.Patch(color='b', label='Detector'),
                              mpatches.Patch(color='yellow', linestyle='--', label='Skin/Foam Boundary')],
           loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05), fontsize=18) # Increased legend fontsize


plt.tight_layout(rect=[0, 0.1, 1, 1]) # Adjust layout to make space for the legend
#plt.suptitle('Geometry Views with Source and Detector Positions', fontsize=18, y=1.02) # Increased suptitle fontsize
plt.show()